In [4]:
#IMPORTS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance

import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf



warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# What Is Actually Making Sofia's Air Dirty?
## A Multi-Predictor Regression Study of PM10 in Sofia, Bulgaria

---

Sofia consistently ranks among the most polluted EU capitals. Every winter, a thick grey smog settles over the city — a visible sign of dangerously high concentrations of **particulate matter (PM)**. The most regulated size fraction is **PM10** — particles under 10 micrometres in diameter — for which the EU sets a legal annual average limit of 40 µg/m³. Sofia routinely exceeds it.

The usual suspects are well known: wood-burning stoves, old diesel cars, industrial activity. But Bulgaria's electricity grid has also changed dramatically — lignite (brown coal) generation collapsed 59% in 2024, with CO₂-free sources now comprising 66% of the power mix (Ember Monthly Electricity Data, 2023).

So here is the question nobody has cleanly answered: **does grid-level decarbonization actually show up in Sofia's air quality (AQ), or is the signal completely buried under heating season and weather?**

To answer it, we can't just plot energy mix against pollution and call it correlation. We have to control for the things that dominate — temperature, wind, the heating season — and then look at what's left. That remainder, if anything, belongs to the grid.

We will:
1. Collect and merge three independent data sources: EEA station-level PM10 measurements, Ember monthly electricity generation data, and ERA5 meteorological reanalysis via Open-Meteo
2. Explore the data and establish the seasonal and meteorological structure of Sofia's PM10 exceedances
3. Build a multiple linear regression model with heating season, wind speed, temperature, and lignite generation share as predictors
4. Quantify the partial effect of the energy mix after controlling for the dominant confounders
5. Cross-validate findings using a Random Forest model and compare feature importances
6. Interpret the result honestly — including if the grid signal doesn't survive the controls

---
## Section 1 — Problem Formulation

### 1.1 The Air Quality Problem

Air pollution in Sofia is fundamentally a **source attribution problem**. PM10 in the air above the city at any given moment is the sum of contributions from multiple independent sources: domestic solid fuel heating, road traffic, industry, and — indirectly — the national electricity grid through its fossil fuel plants.

The challenge is that these sources are correlated with each other and with external factors. Cold days mean more heating *and* more atmospheric stability (reduced vertical mixing, so pollutants accumulate). Winter means both heating season *and* low wind speeds in Sofia's valley basin. A naive regression of PM10 on energy mix would absorb all of these.

The central question is:

> **To what extent does the electricity generation mix influence PM10 levels in Sofia, once we control for heating season and meteorological conditions?**

This is a **causal inference problem cast as a regression problem**. We cannot run a controlled experiment — we can't turn lignite generation on and off while holding everything else constant. What we *can* do is partial out the known confounders and examine whether the energy mix variable retains a statistically significant coefficient in the residual.

### 1.2 Why Multiple Regression?

Multiple linear regression (MLR) is the right starting tool here because it directly answers a question of the form: *what is the effect of variable X on outcome Y, holding variables Z₁, Z₂, ... constant?* The coefficient on lignite share is exactly that partial effect — the change in PM10 associated with a one-unit change in lignite generation share, after removing the contribution of temperature, wind, and season.

This is not the only approach. The tradeoffs between methods are:

| Method | Pros | Cons |
|---|---|---|
| Multiple Linear Regression | Interpretable coefficients, p-values, partial effects | Assumes linearity, sensitive to outliers |
| Random Forest | Captures non-linear interactions, robust | No direct partial effects, black box |
| ARIMA / time series | Handles autocorrelation explicitly | Doesn't naturally incorporate exogenous variables |
| Causal inference (IV, DiD) | Strongest causal claims | Requires valid instruments, complex setup |

We use MLR as the primary method and Random Forest as a cross-validation tool. Both are run on the same feature set, so we can compare their feature importance rankings directly.

### 1.3 Hypotheses

**H₁ (Primary):** Heating season and meteorological variables will explain the majority of variance in daily PM10 concentrations, with lignite generation share retaining a small but statistically significant positive partial effect after controlling for these factors.

**H₂ (Alternative):** The partial effect of lignite generation on PM10 is not statistically significant once heating season and meteorology are controlled for — implying grid-level decarbonization has not produced a detectable signal in urban air quality at the monthly timescale.

**H₀ (Null):** None of the predictor variables has a significant linear relationship with PM10 concentrations in Sofia.

### 1.4 Assumptions & Constraints

Our model makes the following explicit assumptions:
- ERA5 reanalysis data from Open-Meteo is a valid proxy for local meteorological conditions in Sofia (~9 km spatial resolution)
- National-level generation data from Ember is a valid proxy for grid dispatch conditions affecting Sofia — Bulgaria operates as a single bidding zone with no sub-regional breakdown
- Domestic heating intensity is proxied by a binary heating season variable (October–March) and heating degree days derived from temperature — not directly measured
- The relationship between predictors and PM10 is approximately linear at monthly aggregation

**Known limitations:**
- The model cannot directly measure domestic wood/coal burning — the largest single PM10 source in Sofia. This is the primary structural limitation.
- Traffic counts are not included. Day-of-week features partially capture this.
- Causal claims cannot be made from observational regression alone. This is an associational study with controlled confounders.

---
## Section 2 — The Mathematics of Regression with Confounders

### 2.1 Multiple Linear Regression

The core model is:

$$\text{PM10}_t = \beta_0 + \beta_1 \cdot \text{HeatingDays}_t + \beta_2 \cdot \text{WindSpeed}_t + \beta_3 \cdot \text{Temperature}_t + \beta_4 \cdot \text{LigniteShare}_t + \beta_5 \cdot \text{Month}_t + \varepsilon_t$$

Where:
- $\text{PM10}_t$ is the monthly average PM10 concentration at Sofia monitoring stations (µg/m³)
- $\text{HeatingDays}_t$ is the number of heating degree days in month $t$ — a continuous proxy for solid fuel burning intensity
- $\text{WindSpeed}_t$ is the mean monthly wind speed (m/s) — controls for atmospheric dispersion
- $\text{Temperature}_t$ is the mean monthly temperature (°C) — controls for atmospheric stability and heating demand
- $\text{LigniteShare}_t$ is lignite generation as a fraction of total generation (0–1)
- $\text{Month}_t$ is a categorical variable capturing remaining seasonality
- $\varepsilon_t \sim \mathcal{N}(0, \sigma^2)$ is the error term

The coefficient of interest is $\beta_4$. Its sign, magnitude, and p-value answer our research question.

### 2.2 Ordinary Least Squares Estimation

In matrix form, the model is:

$$\mathbf{y} = \mathbf{X}\boldsymbol{\beta} + \boldsymbol{\varepsilon}$$

where $\mathbf{y} \in \mathbb{R}^n$ is the vector of PM10 observations, $\mathbf{X} \in \mathbb{R}^{n \times p}$ is the design matrix of predictors (with a column of ones for the intercept), and $\boldsymbol{\beta} \in \mathbb{R}^p$ is the vector of coefficients.

The OLS estimator minimizes the sum of squared residuals:

$$\hat{\boldsymbol{\beta}} = \underset{\boldsymbol{\beta}}{\arg\min} \left\| \mathbf{y} - \mathbf{X}\boldsymbol{\beta} \right\|^2$$

The closed-form solution is:

$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$$

This requires $\mathbf{X}^\top \mathbf{X}$ to be invertible — which fails under perfect multicollinearity. We check for this using the **Variance Inflation Factor (VIF)**.

### 2.3 Assessing Coefficient Significance

Under the Gauss-Markov assumptions, the OLS estimator is BLUE (Best Linear Unbiased Estimator). The standard error of each coefficient is:

$$\text{SE}(\hat{\beta}_j) = \sqrt{s^2 \left[(\mathbf{X}^\top \mathbf{X})^{-1}\right]_{jj}}$$

where $s^2 = \frac{\|\mathbf{y} - \mathbf{X}\hat{\boldsymbol{\beta}}\|^2}{n - p}$ is the residual variance estimate.

The t-statistic for testing $H_0: \beta_j = 0$ is:

$$t_j = \frac{\hat{\beta}_j}{\text{SE}(\hat{\beta}_j)} \sim t_{n-p}$$

We reject $H_0$ at the $\alpha = 0.05$ level if $|t_j| > t_{n-p, 0.025}$, equivalently if $p\text{-value} < 0.05$.

### 2.4 Model Fit: R² and Adjusted R²

The coefficient of determination $R^2$ measures the proportion of variance in PM10 explained by the model:

$$R^2 = 1 - \frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}} = 1 - \frac{\sum_t (y_t - \hat{y}_t)^2}{\sum_t (y_t - \bar{y})^2}$$

Since $R^2$ increases mechanically with additional predictors, we also report the adjusted $R^2$, which penalises model complexity:

$$\bar{R}^2 = 1 - \frac{(1 - R^2)(n - 1)}{n - p - 1}$$

### 2.5 Variance Inflation Factor (Multicollinearity Check)

Temperature and heating degree days are correlated by construction. The VIF for predictor $j$ measures how much its variance is inflated by collinearity with the other predictors:

$$\text{VIF}_j = \frac{1}{1 - R^2_j}$$

where $R^2_j$ is the $R^2$ from regressing predictor $j$ on all other predictors. A VIF above 10 is typically considered problematic and may warrant dropping or combining collinear features.

---
## Section 3 — Data Collection

### 3.1 Sources

| # | Source | What it provides | Access |
|---|---|---|---|
| 1 | EEA Air Quality e-Reporting Database | Daily PM10 at Sofia stations, 2015–2023 | Free, no registration — `airbase` Python library |
| 2 | Ember Monthly Electricity Data | Monthly generation by fuel type for Bulgaria, 2015–2023 | Free, no registration — direct CSV download |
| 3 | Open-Meteo Historical Weather API | Daily ERA5 temperature, wind speed, precipitation at Sofia (42.70°N, 23.32°E) | Free, no registration — REST API |


The EEA database is the authoritative, validated source for ambient air quality in Europe — the same data used in EU compliance assessments. Ember aggregates Eurostat and ENTSO-E generation data into a clean, citable monthly time series, removing the need for ENTSO-E API registration. Open-Meteo provides ERA5 reanalysis meteorology, which is the global standard for long-period weather reconstruction — consistent, gap-free, and extending back to 1940.